# Date-range calendar: events + promote / cross-sell products

Enter a **from → to** month/year range. This notebook lists every event and entertainment release that overlaps that window, plus the catalog games and attach products you can promote and cross-sell during each event.

Builds on notebooks 01–04 (catalog, promotion plans, trends, event cross-sell). The same lookup powers the Floor Brief **Calendar** page.

Horizon: **2026–2030**, including announced / yet-to-be-released titles and their release windows.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.date_range import calendar_range_payload, format_calendar_range
from src.load_data import load_adaptations, load_catalog, load_events
from src.promote import build_plans

events = load_events()
adaptations = load_adaptations()
catalog = load_catalog(games_only=True, drop_placeholder_dates=True)
plans = build_plans(events, adaptations, catalog)
announced = [row for row in catalog if row.get("product_type") == "announced"]
print(f"events: {len(events)} · adaptations: {len(adaptations)}")
print(f"catalog games: {len(catalog)} · announced / unreleased: {len(announced)}")
print(f"promotion plans: {len(plans)}")


## Enter a month / year range

Change the start and end month/year, then re-run. Example: June–August 2026 covers FIFA World Cup, Summer Game Fest, Gamescom lead-in, and theatrical windows in that span.


In [ ]:
START_YEAR, START_MONTH = 2026, 6
END_YEAR, END_MONTH = 2026, 8
KIND = ""  # "" | "event" | "adaptation"
PRECISION = "dated"  # "exact" = confirmed days · "dated" = day + month · "all" = also quarter/year

payload = calendar_range_payload(
    start_year=START_YEAR,
    start_month=START_MONTH,
    end_year=END_YEAR,
    end_month=END_MONTH,
    events=events,
    adaptations=adaptations,
    plans=plans,
    kind=KIND,
    precision=PRECISION,
    limit=60,
    products_per_event=6,
)
print(format_calendar_range(payload))
print("\nSummary:")
for line in payload["in_short"]:
    print(" -", line)


## Another range: Q4 planning (Oct–Dec 2026)

Awards season, NFL, The Game Awards, winter sales, and late-year announced release windows.


In [ ]:
payload = calendar_range_payload(
    start_year=2026,
    start_month=10,
    end_year=2026,
    end_month=12,
    events=events,
    adaptations=adaptations,
    plans=plans,
    limit=40,
    products_per_event=5,
)
print(f"{payload['event_count']} windows · {payload['unique_products']} unique products\n")
for card in payload["events"][:15]:
    print(f"{card['start']}  {card['name'][:50]}")
    if card.get("hero"):
        print(f"   → promote / cross-sell: {card['hero']} (+{max(0, card['product_count']-1)} more)")


## Single month: September 2026

Every row carries how precise its date is, so a one-month search returns titles whose
real window falls inside that month — including month-precision announcements such as
*Marvel's Wolverine* — instead of year-end placeholders.


In [ ]:
payload = calendar_range_payload(
    start_year=2026,
    start_month=9,
    end_year=2026,
    end_month=9,
    events=events,
    adaptations=adaptations,
    plans=plans,
    precision="dated",
    limit=60,
    products_per_event=6,
)
print(f"{payload['event_count']} windows · {payload['exact_count']} with a confirmed day")
print("precision mix:", payload["precision_mix"], "\n")
for card in payload["events"]:
    marker = "" if card["exact_date"] else f"  [{card['date_precision']} window]"
    print(f"{card['date_label'] or card['start']}{marker}  {card['name']}")
    for item in card["products"][:3]:
        print(f"      · {item['canonical_title']} ({item['role']})")
